# Тестирование и качество кода

Программа может работать на одном примере и ошибаться на другом. Тесты помогают проверять поведение программы на знакомых входных данных. Понятные имена, небольшие функции и единый стиль упрощают чтение и исправление кода.

> Запускай ячейки по порядку. В этом уроке мы будем писать и запускать простые тесты.

## Карта урока

1. Что такое тест
2. Понятные имена и форматирование
3. Разбиваем программу на небольшие функции
4. Проверяем результат через `assert`
5. Пишем автоматические тесты с `unittest`
6. Проверяем граничные случаи
7. Практика и памятка


## 1. Что такое тест

**Тест** проверяет, что программа ведёт себя ожидаемым образом для заданных данных. Для функции обычно известны вход и правильный результат.

Например, если функция складывает цены, можно заранее подобрать случаи:

| Входные цены | Ожидаемая сумма |
|---|---:|
| `[100, 50]` | `150` |
| `[0, 0]` | `0` |
| `[]` | `0` |

![Большая задача делится на маленькие функции, каждую можно проверить](assets/small_functions_tests.png)


## 2. Понятные имена и форматирование

Сравни две записи. В первой трудно понять, что делают переменные. Во второй имена объясняют смысл.


In [ ]:
x = 3
p = 450
r = x * p
print(r)

number_of_items = 3
price_per_item = 450
total_price = number_of_items * price_per_item
print(total_price)

Полезные привычки:

- называй переменные и функции по их назначению: `total_price`, `calculate_total`;
- выбирай имена, которые легко прочитать и произнести;
- для переменных и функций используй `snake_case`;
- делай отступы в четыре пробела;
- отделяй определения функций пустыми строками;
- избегай очень длинных строк и лишних пробелов.

Python не требует именно таких имён, но единый стиль помогает людям читать код. Эти соглашения описаны в руководстве PEP 8.


### Сделаем код понятнее

Измени названия `a`, `b` и `c`, чтобы по коду было ясно, что он считает.


In [ ]:
a = 8
b = 5
c = a * b
print(c)

## 3. Разбиваем программу на небольшие функции

Функцию проще проверить, если у неё одна понятная задача: например, посчитать сумму, применить скидку или отформатировать строку.

Сначала посмотрим на программу, где ввод, вычисление и вывод смешаны вместе.


In [ ]:
prices = [120, 80, 50]
total = 0
for price in prices:
    total += price

print(f"Сумма покупок: {total} рублей")

Вынесем вычисление в отдельную функцию. Теперь её можно проверять независимо от ввода и печати.


In [ ]:
def calculate_total(prices):
    total = 0
    for price in prices:
        total += price
    return total


prices = [120, 80, 50]
total = calculate_total(prices)
print(f"Сумма покупок: {total} рублей")

Функция `calculate_total()` считает. Отдельный код решает, откуда взять цены и как показать сумму. Такое разделение помогает находить, какая именно часть требует исправления.

## 4. Проверяем результат через `assert`

`assert` проверяет условие. Если оно истинно, программа продолжает работу. Если ложно, Python сообщает об `AssertionError`.


In [ ]:
assert calculate_total([100, 50]) == 150
assert calculate_total([0, 0]) == 0
assert calculate_total([]) == 0

print("Все проверки прошли")

Если функция случайно начинает возвращать неверное значение, одна из проверок остановится. Добавь тест для каждого важного случая, который хочешь защитить.

**Важно:** `assert` подходит для проверок во время разработки. Не используй его как единственную проверку пользовательского ввода: в некоторых режимах запуска Python может отключать `assert`.


### Сверяем ожидаемое и фактическое значение

Перед проверкой полезно знать три вещи: входные данные, ожидаемый результат и фактический результат.


In [ ]:
prices = [20, 30, 40]
expected_total = 90
actual_total = calculate_total(prices)

print("Ожидали:", expected_total)
print("Получили:", actual_total)
assert actual_total == expected_total

## 5. Автоматические тесты с `unittest`

Модуль `unittest` входит в стандартную библиотеку Python. Тесты можно объединить в класс. Имя класса начинается с `Test`, а имя каждого тестового метода — с `test_`.

Метод `assertEqual()` сравнивает ожидаемое и фактическое значения и показывает их, если проверка не прошла.


In [ ]:
import unittest


class TestCalculateTotal(unittest.TestCase):
    def test_adds_prices(self):
        self.assertEqual(calculate_total([100, 50]), 150)

    def test_empty_list_is_zero(self):
        self.assertEqual(calculate_total([]), 0)

    def test_zero_prices(self):
        self.assertEqual(calculate_total([0, 0]), 0)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalculateTotal)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

Когда тестов становится больше, их часто хранят отдельно от программы, например в файле `test_calculations.py`. Из корневой папки проекта тесты можно запустить командой:

```text
python -m unittest
```

По соглашению тестовые файлы и функции начинаются с `test_`, чтобы их было удобно находить автоматически.


### Тестируем неверные значения

Если функция должна сообщать об ошибке для недопустимых данных, тест можно написать с `assertRaises()`.


In [ ]:
def calculate_discount(price, percent):
    if price < 0:
        raise ValueError("Цена не может быть отрицательной")
    if not 0 <= percent <= 100:
        raise ValueError("Скидка должна быть от 0 до 100 процентов")
    return price * (1 - percent / 100)


class TestCalculateDiscount(unittest.TestCase):
    def test_valid_discount(self):
        self.assertEqual(calculate_discount(100, 25), 75)

    def test_negative_price_is_rejected(self):
        with self.assertRaises(ValueError):
            calculate_discount(-1, 10)

    def test_discount_over_one_hundred_is_rejected(self):
        with self.assertRaises(ValueError):
            calculate_discount(100, 101)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalculateDiscount)
unittest.TextTestRunner(verbosity=2).run(suite)

### Несколько полезных проверок `unittest`

- `assertEqual(actual, expected)` — значения равны;
- `assertTrue(condition)` и `assertFalse(condition)` — условие истинно или ложно;
- `assertIn(item, collection)` — элемент входит в коллекцию;
- `assertRaises(ErrorType)` — код поднимает ожидаемую ошибку;
- `assertAlmostEqual(actual, expected)` — приблизительное сравнение дробных чисел.


In [ ]:
class TestSmallChecks(unittest.TestCase):
    def test_membership(self):
        self.assertIn("Python", ["Python", "код"])

    def test_boolean_result(self):
        self.assertTrue(5 > 2)
        self.assertFalse(1 > 2)

    def test_fraction(self):
        self.assertAlmostEqual(0.1 + 0.2, 0.3)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestSmallChecks)
unittest.TextTestRunner(verbosity=2).run(suite)

## 6. Проверяем граничные случаи

Тестируй не только обычный пример. Подумай о границах и необычных входных данных:

- пустой список;
- одно значение;
- ноль;
- минимальное и максимальное допустимые значения;
- некорректный ввод.

Не нужно проверять каждое возможное значение. Выбирай случаи, которые подтверждают важные правила программы.


In [ ]:
def count_long_words(words, minimum_length):
    count = 0
    for word in words:
        if len(word) >= minimum_length:
            count += 1
    return count


assert count_long_words(["кот", "черепаха", "ёж"], 4) == 1
assert count_long_words([], 4) == 0
assert count_long_words(["кот"], 3) == 1
print("Проверки количества слов прошли")

Список тестовых случаев стоит превратить в автоматические проверки, если функция важна и её легко случайно сломать.

### Ошибка в тесте или в программе?

Если тест не прошёл, это ещё не означает, что ошибочна именно функция. Проверь, правильно ли записано ожидаемое значение и соответствует ли оно условию задачи.


## 7. Практика

### Задание 1. Осмысленные имена

Переименуй переменные `x`, `y` и `z` в примере так, чтобы было ясно, что он считает.


In [ ]:
x = [12, 18, 20]
y = 0
for item in x:
    y += item
z = y / len(x)
print(z)

### Задание 2. Небольшие функции

Раздели программу на две функции: одну для вычисления суммы покупок, другую для применения скидки. Функции должны возвращать результат, а не печатать его.


In [ ]:
prices = [100, 50, 25]
discount_percent = 10

# Напиши функции и проверь получившуюся цену здесь


### Задание 3. Автоматические тесты

Напиши функцию `is_adult(age)`, которая возвращает `True`, если возраст не меньше 18, и `False` иначе. Создай для неё класс `TestIsAdult` с тестами для возраста 18, 17 и 0.


In [ ]:
# Напиши функцию и unittest-класс здесь


## Домашние задания

### Базовый уровень

1. Напиши функцию, которая возвращает самое большое число из списка. Добавь тесты для обычного списка и списка из одного числа.
2. Создай функцию проверки чётности. Напиши тесты для чётного, нечётного и нулевого значения.
3. Перепиши длинные и неясные имена переменных так, чтобы назначение кода было понятно без комментариев.

### Средний уровень

4. Напиши функцию, которая считает среднюю оценку. Отдельно продумай, что делать с пустым списком.
5. Раздели программу подсчёта покупок на функции: чтение данных, вычисление суммы, применение скидки и форматирование результата.
6. Напиши автоматические тесты на успешный расчёт скидки и на недопустимую цену.

### Задание со звёздочкой ★

Возьми один из небольших проектов курса. Выдели в нём несколько функций с одной задачей и напиши для каждой минимум по два теста: обычный случай и граничный.


## Короткое повторение

- Тест сравнивает ожидаемый результат с фактическим.
- `assert` подходит для простых проверок во время разработки.
- `unittest` позволяет группировать проверки и запускать их автоматически.
- Тестовые методы обычно начинаются с `test_`.
- Проверь обычные, пустые, граничные и ошибочные случаи.
- Понятные имена и небольшие функции упрощают чтение и тестирование.
- Отделяй вычисление от ввода и вывода, когда это делает код яснее.

### Проверь себя

1. Что проверяет `assert`?
2. Зачем функциям в тестируемом коде лучше возвращать результат?
3. С чего начинается имя метода `unittest`, который должен запускаться как тест?
4. Какие граничные случаи стоит проверить для функции работы со списком?
5. Почему `assert` не следует использовать для единственной проверки пользовательского ввода?
